In [1]:
from TableTennisEnvironmentV0.TableTennisEnvironment import TableTennisEnv
import pybullet as p 

env = TableTennisEnv(show_gui=False)

pybullet build time: Feb  4 2024 12:55:26


In [2]:
## We need to estimate the initial velocity of the ball 
## although we can get it in pybullet, we won't get it directly in optitrack

def estimateInitVelocity(ball_position1, ball_position2, time_gap):
    displacement = (ball_position2[0] - ball_position1[0], ball_position2[1] - ball_position1[1], ball_position2[2] - ball_position1[2])
    velocity = [displacement[i] / time_gap for i in range(len(displacement))]
    return velocity

## Let's test it
## This is real initial velocity
u_x, u_y, u_z = 5.5, 0.8, -3     # in meters/sec
real_initial_velocity = [u_x, u_y, u_z]
env.throw_ball(real_initial_velocity)

## now we estimate the initial velocity
ball_position1 = p.getBasePositionAndOrientation(env._ball)[0]
action = [0, 0, 0, 0, 0, 0]
env.step(action)
ball_position2 = p.getBasePositionAndOrientation(env._ball)[0]
time_gap = 1/240

estimated_initial_velocity = estimateInitVelocity(ball_position1, ball_position2, time_gap)

print('real velocity: ', real_initial_velocity)
print('estimated velocity: ', estimated_initial_velocity)



real velocity:  [5.5, 0.8, -3]
estimated velocity:  [5.493293801502528, 0.7990245529458223, -3.0372170735468096]


So the results are pretty accurate

![Alt text](illustration.jpeg)


In [3]:
from utils.math_solvers import solve_quadratic

# dimensions of the table
[Lx, Ly, Lz] = [2.74, 1.5, 1.0]


# distance between ball and the robot
ball_initial_position = p.getBasePositionAndOrientation(env._ball)[0] # can be collected from optitrack
robot_initial_position = p.getLinkState(env._robotic_arm, 0)[0]      # can be collected from optitrack
distance_x, distance_y, distance_z = (robot_initial_position[0] - ball_initial_position[0], robot_initial_position[1] - ball_initial_position[1], robot_initial_position[2] - ball_initial_position[2])

# height of the ball from the table
h1 = ball_initial_position[2] - Lz - 0.2 # 0.2m is the height of the red surface


# now calculate the time for the contact with ground
g = 9.8
a = -0.5*g
b = estimated_initial_velocity[2]
c = h1

t1 = None # it is the time for the ball to hit the ground for the first time

roots = solve_quadratic(a, b, c)
for root in roots:
    if root.imag == 0 and root.real > 0: # means complex root, ignore
        t1 = root.real


## now how much distance did the 
d1_x = estimated_initial_velocity[0]*t1
d1_y = estimated_initial_velocity[1]*t1


## Velocities at this position
vx = estimated_initial_velocity[0]
vy = estimated_initial_velocity[1]
vz = estimated_initial_velocity[2] - g*t1


## remaining distance that needs to be covered to reach the robot - maybe we need to set constraints on y displacement as well
d2_x = distance_x - d1_x

new_velocity = [vx, vy, -vz] # vz will change sign because now the direction of the projectile in z axis is reversed


## not calculate how high the projectile can go 
t2 = distance_x / estimated_initial_velocity[0] - t1 # remaining flight time to reach the robot
h2 = new_velocity[2]*t2 - 0.5*g*(t2**2)


# now recompute the position of the ball on the Y-Z plane

y = ball_initial_position[1] + estimated_initial_velocity[1] * (t1 + t2)
z = h2 + Lz + 0.2

estimated_hitting_point = (y, z)
print('total flight time: ', t1+t2, 's')
print('The ball will hit the Y-Z plane at: ', estimated_hitting_point)

total flight time:  0.487433502117562 s
The ball will hit the Y-Z plane at:  (0.3927974256909091, 2.22927586372358)


In [4]:
import time
def ball_crossed_yz(ball_current_position, robot_current_position):#this function will detect if the ball reached Y-Z plane
    ball_current_position = p.getBasePositionAndOrientation(env._ball)[0] # can be collected from optitrack
    x_ball, y_ball, z_ball = ball_current_position
    x_robot, y_robot, z_robot = robot_current_position

    if x_ball > x_robot:  # this condition ensures the ball crossed Y-Z plane
        return True

    else:
        return False


def convert_step_to_time(step_number, frequency = 240):
    return step_number/frequency

action = [0, 0, 0, 0, 0, 0] # keep the robot stopped, we don't need it.

for step in range(5000):
    env.step(action)
    ball_current_position = p.getBasePositionAndOrientation(env._ball)[0]
    if ball_crossed_yz(ball_current_position, robot_initial_position):
        print(convert_step_to_time(step, 240))
        break
    time.sleep(1/240) # because the simulation is at 240 Hz

real_hitting_point = (p.getBasePositionAndOrientation(env._ball)[0][1], p.getBasePositionAndOrientation(env._ball)[0][2])
print('The ball has hit the Y-Z plane at: ', real_hitting_point)

# now calculate delta

0.725
The ball has hit the Y-Z plane at:  (0.3670424977504225, 1.4559086300728026)


In [8]:
## calculate delta

delta = (real_hitting_point[0] - estimated_hitting_point[0], real_hitting_point[1] - estimated_hitting_point[1])
print('delta: ', delta)

delta:  (-0.02575492794048656, -0.7733672336507775)
